# Stage 2 Notebook 48 - Exp2SS Full 70K dataset + VFL on anchor head

**Test the data-scale hypothesis with the FIXED loss recipe.** The dataset visualization confirmed 70K training samples are healthy (mean 5.81 lanes/image, max_lanes=10 truncates 7.8%). Earlier full-dataset runs (NB28 / NB38) used the broken alpha=0.25 ASL recipe and showed lane plateau + det degradation.

Exp2SS retries the data scale with the Exp2QQ VFL+IoU regression recipe so we can compare:
- 3000 samples + VFL: Exp2QQ (NB46)
- 70K samples + VFL: Exp2SS (this notebook)

If VFL + 70K data unlocks the cls signal, decoded_f1 should reach >= 0.30 (CLRKDNet small-CULane territory). If 70K + VFL doesn't help past Exp2QQ, the bottleneck is architecture, not data.

70K images / 8 batch = 8750 iter/epoch. 6 epochs * 8750 = 52500 iters total (vs 7500 in 20-epoch 3K runs). That's 7x more iter and each iter sees 23x more variety. Wall-clock ~ 60-80 minutes on RTX Pro 6000.

Config:
- Same as Exp2QQ (anchor + VFL + IoU regression).
- `train.end_epoch: 20 -> 6` (we get the same total loss steps from full data).
- LIMIT_TRAIN flag REMOVED in the train cell -> uses full 70K split.

### Run mode

1. `DEBUG_MODE = True` smoke first.
2. `DEBUG_MODE = False` for the full-dataset run. Note `LIMIT_TRAIN = None` is the toggle.
3. Wall-clock ~ 60-80 min on RTX Pro 6000.
4. Independent of all prior NBs except for shared codebase.
5. If you hit OOM, drop batch_size to 6.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp43_rmt_gca_anchor_vfl_full_dataset_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp43_rmt_gca_anchor_vfl_full_dataset_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp43_rmt_gca_anchor_vfl_full_dataset_joint_smoke.log
OK exp43_rmt_gca_anchor_vfl_full_dataset_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=4.9071 det_loss=3.0840 grad_cos=0.1325 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.4992026388645172, 'gate/lane_mean': 0.49619460105895996, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp43_rmt_gca_anchor_vfl_full_dataset_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'full6'
    EPOCHS = 6
    BATCH_SIZE = 8
    LIMIT_TRAIN = None
    LIMIT_VAL = 2000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: None
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp43_rmt_gca_anchor_vfl_full_dataset_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp43_rmt_gca_anchor_vfl_full_dataset_joint_full6 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp43_rmt_gca_anchor_vfl_full_dataset_joint_full6.tar --epochs 6 --batch-size 8 --limit-val 2000 --force-extract --print-every 50
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp43_rmt_gca_anchor_vfl_full_dataset_joint_full6.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp43_rmt_gca_anchor_vfl_full_dataset_joint_full6_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp43_rmt_gca_anchor_vfl_full_dataset_joint.yaml --curve-tar /content/driv

0

## What to watch in Exp2SS training

Pass criteria at epoch 6:
- **`val/lane/decoded_f1 >= 0.20`** -- 5x NB40, beating Exp2QQ if data scale matters.
- **`val/matched_line_iou >= 0.50`** -- with 23x more data and proper VFL, geometry should be near the project ceiling (NB45 hit 0.525 with width 1.0; Exp2SS with width 0.5 + 70K data should match).
- **`val/lane/decoded_oracle_f1 >= 0.45`**.
- `val_det` should DECREASE monotonically (in NB38 it INCREASED from 2.07 to 2.77 -- a sign of joint conflict that should NOT recur with VFL since the lane gradient is now in a non-degenerate regime).

Failure signals:
- decoded_f1 < 0.10 even with 70K data: the cls representation bottleneck is intrinsic to the 192-anchor design. Exp2RR's K=64 query head is the right path.
- val_det INCREASES like NB38 did: joint conflict is unrelated to cls. Lower lambda_det to 0.5 in a follow-up experiment.
- OOM at full dataset: drop batch_size to 6 (`--batch-size 6`).